In [22]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
from pydub import AudioSegment
import simpleaudio as sa
import requests
import os
from io import BytesIO
from dotenv import load_dotenv
from typing import Optional
import re
import time

load_dotenv()
# ------------------------------
# 1. Setup Authentication
# ------------------------------
client_id = os.getenv("CLIENT_ID")
client_secret = os.getenv("CLIENT_SECRET")

sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(
    client_id=client_id,
    client_secret=client_secret
))

# ------------------------------
# 2. Search for artist and get tracks
# ------------------------------
def get_spotify_preview_url(spotify_track_id: str) -> Optional[str]:
    """
    Get the preview URL for a Spotify track using the embed page workaround.
    """
    try:
        embed_url = f"https://open.spotify.com/embed/track/{spotify_track_id}"
        response = requests.get(embed_url)
        response.raise_for_status()
        html = response.text
        match = re.search(r'"audioPreview":\s*{\s*"url":\s*"([^"]+)"', html)
        return match.group(1) if match else None
    except Exception as e:
        print(f"Failed to fetch preview URL from embed: {e}")
        return None

# ------------------------------
# 3. Search for artist and get track list
# ------------------------------
def get_tracks(artist_name, limit=10):
    results = sp.search(q=f'artist:{artist_name}', type='track', limit=limit)
    items = results['tracks']['items']
    print(results['tracks'])
    print(len(results['tracks']))
    tracks = []
    for t in items:
        preview_url = t['preview_url']
        if not preview_url:
            preview_url = get_spotify_preview_url(t['id'])  # Try workaround
            time.sleep(1)
        if preview_url:
            tracks.append({
                'name': t['name'],
                'artist': t['artists'][0]['name'],
                'preview_url': preview_url,
                'id': t['id']
            })
    return tracks, results

# Example usage:
tracks, results = get_tracks('Linkin Park', limit=5)
for i, t in enumerate(tracks):
    print(f"{i+1}. {t['name']} by {t['artist']}")


{'href': 'https://api.spotify.com/v1/search?offset=0&limit=5&query=artist%3ALinkin%20Park&type=track', 'limit': 5, 'next': 'https://api.spotify.com/v1/search?offset=5&limit=5&query=artist%3ALinkin%20Park&type=track', 'offset': 0, 'previous': None, 'total': 127, 'items': [{'album': {'album_type': 'album', 'artists': [{'external_urls': {'spotify': 'https://open.spotify.com/artist/6XyY86QOPPrYVGvF9ch6wz'}, 'href': 'https://api.spotify.com/v1/artists/6XyY86QOPPrYVGvF9ch6wz', 'id': '6XyY86QOPPrYVGvF9ch6wz', 'name': 'Linkin Park', 'type': 'artist', 'uri': 'spotify:artist:6XyY86QOPPrYVGvF9ch6wz'}], 'available_markets': ['AR', 'AU', 'AT', 'BE', 'BO', 'BR', 'BG', 'CL', 'CO', 'CR', 'CY', 'CZ', 'DK', 'DO', 'DE', 'EC', 'EE', 'SV', 'FI', 'FR', 'GR', 'GT', 'HN', 'HK', 'HU', 'IS', 'IE', 'IT', 'LV', 'LT', 'LU', 'MY', 'MT', 'MX', 'NL', 'NZ', 'NI', 'NO', 'PA', 'PY', 'PE', 'PH', 'PL', 'PT', 'SG', 'SK', 'ES', 'SE', 'CH', 'TW', 'TR', 'UY', 'US', 'GB', 'AD', 'LI', 'MC', 'ID', 'JP', 'TH', 'VN', 'RO', 'IL', '

In [26]:
results["tracks"]["items"]

[{'album': {'album_type': 'album',
   'artists': [{'external_urls': {'spotify': 'https://open.spotify.com/artist/6XyY86QOPPrYVGvF9ch6wz'},
     'href': 'https://api.spotify.com/v1/artists/6XyY86QOPPrYVGvF9ch6wz',
     'id': '6XyY86QOPPrYVGvF9ch6wz',
     'name': 'Linkin Park',
     'type': 'artist',
     'uri': 'spotify:artist:6XyY86QOPPrYVGvF9ch6wz'}],
   'available_markets': ['AR',
    'AU',
    'AT',
    'BE',
    'BO',
    'BR',
    'BG',
    'CL',
    'CO',
    'CR',
    'CY',
    'CZ',
    'DK',
    'DO',
    'DE',
    'EC',
    'EE',
    'SV',
    'FI',
    'FR',
    'GR',
    'GT',
    'HN',
    'HK',
    'HU',
    'IS',
    'IE',
    'IT',
    'LV',
    'LT',
    'LU',
    'MY',
    'MT',
    'MX',
    'NL',
    'NZ',
    'NI',
    'NO',
    'PA',
    'PY',
    'PE',
    'PH',
    'PL',
    'PT',
    'SG',
    'SK',
    'ES',
    'SE',
    'CH',
    'TW',
    'TR',
    'UY',
    'US',
    'GB',
    'AD',
    'LI',
    'MC',
    'ID',
    'JP',
    'TH',
    'VN',
    'RO',
   

In [12]:
import random
def play_preview(preview_url):
    if not preview_url:
        print("No preview available.")
        return
    response = requests.get(preview_url)
    audio = AudioSegment.from_file(BytesIO(response.content), format="mp3")
    play_obj = sa.play_buffer(
        audio.raw_data,
        num_channels=audio.channels,
        bytes_per_sample=audio.sample_width,
        sample_rate=audio.frame_rate
    )
    play_obj.wait_done()

# ------------------------------
# 5. Play random snippet
# ------------------------------
def play_random_snippet(preview_url, snippet_length_ms=5000):
    if not preview_url:
        print("No preview available.")
        return
    response = requests.get(preview_url)
    audio = AudioSegment.from_file(BytesIO(response.content), format="mp3")

    if len(audio) < snippet_length_ms:
        snippet = audio
    else:
        start = random.randint(0, len(audio) - snippet_length_ms)
        snippet = audio[start:start + snippet_length_ms]
    
    play_obj = sa.play_buffer(
        snippet.raw_data,
        num_channels=snippet.channels,
        bytes_per_sample=snippet.sample_width,
        sample_rate=snippet.frame_rate
    )
    play_obj.wait_done()

In [14]:
play_preview(tracks[0]['preview_url'])

In [13]:
play_random_snippet(tracks[0]['preview_url'])